In [6]:
from pathlib import Path
import pickle
from generate_gt_pairs import load_query_cams, generate_gt_for_query
from visual_sfm_3d import visualize_2d3d_matches

In [7]:
output_dir = Path("/proj/vlarsson/outputs") 
scene = "0004"
query_path = output_dir / "query_sets" / scene
query_names = query_path / "query_image_names.txt"
query_pose = query_path / "query_image_cameras.txt"

feats_3d_path = output_dir / "midterm_results" / scene / "points3D_feats_cache.h5" # averaged descriptors for all 3D points
feats_2d_path = output_dir / "sfm" / scene / "feats-superpoint-n2048.h5" # cached SP descriptors
covisibility_result_path = output_dir / "midterm_results" / scene / "covisibility_results.pkl" # covisibility results for all queries
depth_path = Path("/proj/vlarsson/datasets/megadepth/depth_undistorted") / scene # depth maps for all queries

In [8]:
# load query names to a list
with open(query_names, 'r') as f:
    query_list = [line.strip() for line in f]

# load covisibility result, where covisibility_results[query_image] = {'unique_images': set of img_ids,
# 'unique_points': np.array of point3D ids, 'max_distance': float}
with open(covisibility_result_path, "rb") as f:
    covisibility_dict = pickle.load(f)

# load query pose infos
query_cams = load_query_cams(query_pose)

In [9]:
gt_data = {}
for query in query_list[:1]:
    gt_data[query] = generate_gt_for_query(
        query, feats_2d_path, feats_3d_path, query_cams, covisibility_dict, depth_path
        )

Collected descriptors for query image 2732395955_36fffb85ec_o.jpg.
Collected descriptors for 4765 3D points.


In [10]:
current_idx = 0
query_name = query_list[current_idx]
query_image_path = Path("/proj/vlarsson/datasets/megadepth/Undistorted_SfM") / scene / "images" / query_name
fig = visualize_2d3d_matches(
        gt_data[query_name], query_name, query_image_path, query_cameras=query_cams[query_name], html_dir=None, plane_distance=0.5, save_html=False
        )